# DomiSmart — Entrega 1: conexión a Supabase desde Colab

Este notebook crea el esquema de DomiSmart en Supabase (PostgreSQL + pgvector) y verifica que quedó bien.

**Antes de ejecutarlo (solo una vez por grupo):**

1. Crear un proyecto en [supabase.com](https://supabase.com) (plan gratuito). Guardar la contraseña de la base de datos.
2. En el proyecto: botón **Connect** → copiar la cadena de conexión del **Session pooler** (no la *Direct connection*). Tiene esta forma:
   `postgresql://postgres.<ref-del-proyecto>:<CONTRASEÑA>@aws-0-<region>.pooler.supabase.com:5432/postgres`
3. En Colab: ícono de la llave 🔑 (**Secrets**) → **Add new secret** → nombre `DATABASE_URL`, valor = la cadena anterior con la contraseña real → activar *Notebook access*.

⚠️ **Trampa:** la *Direct connection* de Supabase usa IPv6 y Colab no lo soporta, así que falla con un error de conexión. Usen siempre el **Session pooler**.

⚠️ Nunca escriban la contraseña dentro del notebook: si lo suben a GitHub queda pública.

## 1. Instalar librerías

In [ ]:
!pip install -q "psycopg[binary]" pgvector

## 2. Leer la cadena de conexión desde los Secrets de Colab

In [ ]:
import os

try:
    from google.colab import userdata          # en Colab
    DATABASE_URL = userdata.get("DATABASE_URL")
except ImportError:                             # fuera de Colab: variable de entorno
    DATABASE_URL = os.environ["DATABASE_URL"]

print("Cadena cargada:", DATABASE_URL.split("@")[-1])   # muestra solo el host, nunca la contraseña

## 3. Conectarse y revisar versiones

In [ ]:
import psycopg

# prepare_threshold=None evita errores de "prepared statement" si usan el pooler de transacciones (puerto 6543)
conn = psycopg.connect(DATABASE_URL, autocommit=True, prepare_threshold=None)

print(conn.execute("SELECT version()").fetchone()[0])
print("pgvector disponible:",
      conn.execute("SELECT default_version FROM pg_available_extensions WHERE name = 'vector'").fetchone())

## 4. Crear el esquema

Suban `schema.sql` cuando se lo pida (o clonen el repositorio del grupo en Colab).

⚠️ **El script borra y vuelve a crear las 24 tablas.** Ejecuten esta celda solo para crear el esquema la primera vez o cuando lo cambien: en la Entrega 2, cuando ya haya datos cargados, esta celda los borraría.

In [ ]:
import os

if not os.path.exists("schema.sql"):
    from google.colab import files
    files.upload()                               # seleccionar schema.sql

with open("schema.sql", encoding="utf-8") as f:
    conn.execute(f.read())                       # sin parámetros, psycopg acepta el script completo

print("Esquema ejecutado")

## 5. Verificar el esquema

In [ ]:
tablas = [r[0] for r in conn.execute(
    "SELECT tablename FROM pg_tables WHERE schemaname = 'public' ORDER BY tablename")]
print(len(tablas), "tablas (se esperan 24)")
print(", ".join(tablas))

print("\nExtensión vector:",
      conn.execute("SELECT extversion FROM pg_extension WHERE extname = 'vector'").fetchone()[0])

sin_rls = conn.execute(
    "SELECT count(*) FROM pg_tables WHERE schemaname = 'public' AND NOT rowsecurity").fetchone()[0]
print("Tablas sin RLS (se espera 0):", sin_rls)

print("\nÍndices vectoriales:")
for nombre, definicion in conn.execute(
        "SELECT indexname, indexdef FROM pg_indexes WHERE indexdef LIKE '%hnsw%'"):
    print(" -", nombre)

assert len(tablas) == 24 and sin_rls == 0, "Algo no quedó bien: revisen el mensaje de error de la celda 4"
print("\n✅ Esquema correcto")

## 6. Prueba rápida de vectores (no deja datos)

Inserta datos ficticios, hace una búsqueda híbrida texto→imagen y **deshace todo** al final con un rollback.

In [ ]:
import numpy as np
from pgvector.psycopg import register_vector

register_vector(conn)

try:
    with conn.transaction():
        cur = conn.cursor()
        cur.execute("INSERT INTO usuarios (nombre, email) VALUES ('Prueba', 'prueba@ejemplo.com') RETURNING id_usuario")
        id_usuario = cur.fetchone()[0]
        cur.execute("INSERT INTO ciudades (nombre, departamento) VALUES ('Ciudad prueba', 'Depto') RETURNING id_ciudad")
        id_ciudad = cur.fetchone()[0]
        cur.execute("INSERT INTO restaurantes (nombre, tipo_cocina) VALUES ('Rest prueba', 'rápida') RETURNING id_restaurante")
        id_rest = cur.fetchone()[0]
        cur.execute("INSERT INTO sedes (id_restaurante, id_ciudad, nombre, direccion, hora_apertura, hora_cierre) "
                    "VALUES (%s, %s, 'Sede', 'Calle 1', '10:00', '22:00') RETURNING id_sede", (id_rest, id_ciudad))
        id_sede = cur.fetchone()[0]
        cur.execute("INSERT INTO categorias_plato (nombre) VALUES ('Categoría prueba') RETURNING id_categoria")
        id_cat = cur.fetchone()[0]
        cur.execute("INSERT INTO platos (id_restaurante, id_categoria, nombre, descripcion) "
                    "VALUES (%s, %s, 'Plato prueba', 'desc') RETURNING id_plato", (id_rest, id_cat))
        id_plato = cur.fetchone()[0]
        cur.execute("INSERT INTO oferta_platos VALUES (%s, %s, %s, 22000, true)", (id_sede, id_plato, id_rest))
        cur.execute("INSERT INTO documentos (id_restaurante, titulo, tipo_documento, fecha, fuente, licencia, contenido_texto) "
                    "VALUES (%s, 'Doc', 'menu', CURRENT_DATE, 'prueba', 'prueba', 'texto') RETURNING id_doc", (id_rest,))
        id_doc = cur.fetchone()[0]
        cur.execute("INSERT INTO imagenes (id_doc, id_plato, ruta_archivo, descripcion, autor_fuente, licencia) "
                    "VALUES (%s, %s, 'prueba.jpg', 'foto', 'grupo', 'propia') RETURNING id_img", (id_doc, id_plato))
        id_img = cur.fetchone()[0]

        # Vectores aleatorios de 512 dimensiones (en la Entrega 2 vendrán de CLIP)
        vec_img = np.random.rand(512).astype(np.float32)
        cur.execute("INSERT INTO embeddings_imagen (id_img, vector_embedding, modelo) VALUES (%s, %s, 'prueba')",
                    (id_img, vec_img))

        # Búsqueda híbrida: similitud coseno + filtro SQL de precio
        cur.execute("""
            SELECT p.nombre, o.precio, 1 - (ei.vector_embedding <=> %s) AS similitud
            FROM embeddings_imagen ei
            JOIN imagenes i       ON i.id_img = ei.id_img
            JOIN platos p         ON p.id_plato = i.id_plato
            JOIN oferta_platos o  ON o.id_plato = p.id_plato AND o.disponible
            WHERE o.precio < 25000
            ORDER BY ei.vector_embedding <=> %s
            LIMIT 5""", (vec_img, vec_img))
        print("Resultado de la búsqueda híbrida:", cur.fetchall())

        # La base debe rechazar un vector de dimensión equivocada
        try:
            with conn.transaction():
                cur.execute("INSERT INTO embeddings_imagen (id_img, vector_embedding, modelo) VALUES (%s, %s, 'mal')",
                            (id_img, np.random.rand(384).astype(np.float32)))
        except psycopg.errors.DataException as e:
            print("Rechazado como se esperaba:", str(e).strip())

        raise psycopg.Rollback()      # deshace todos los INSERT de esta celda
finally:
    quedan = conn.execute("SELECT count(*) FROM restaurantes").fetchone()[0]
    print("Filas que quedaron en restaurantes (se espera 0):", quedan)

## Siguiente paso (Entrega 2)

Con la conexión lista, la Entrega 2 agrega en este mismo notebook la carga de datos, el chunking con las tres estrategias, los embeddings (MiniLM y CLIP, aprovechando la GPU de Colab: *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*), el pipeline RAG y la evaluación RAGAS.

In [ ]:
conn.close()